# 05 - Audio Pipeline: Preprocessing & Wav2Vec2 Fine-Tuning

**Stage:** Audio-based fake news detection — final model

**What this notebook does:**
- Analyzes and standardizes audio clip lengths (segmenting/padding to 5 seconds)
- Exploratory audio analysis: waveform comparison, spectrograms, amplitude histograms, FFT
- Converts audio to `input_values` via `Wav2Vec2Processor`
- Fine-tunes two separate Wav2Vec2 models: one for Arabic, one for English audio authenticity classification (real vs AI-generated)
- Evaluates with confusion matrix, ROC/AUC, and classification report

**Input:** balanced real/fake audio dataset (real recordings + TTS-generated clips from multiple engines)
**Output:** two fine-tuned Wav2Vec2 models, consumed by `06_audio_api_deployment.ipynb`

> Part of the WAVE project — AI section (audio-based fake news detection).

# **كود التحليل الكامل لطول الأصوات**

In [ ]:
# --- Environment configuration ---
# Set BASE_DIR to wherever your data/model checkpoints live locally.
# If running on Google Colab, uncomment the two lines below.
# from google.colab import drive
# drive.mount('/content/drive')

import os
BASE_DIR = os.environ.get("WAVE_DATA_DIR", "./data")


In [ ]:
import pandas as pd
import os
import librosa
import matplotlib.pyplot as plt


df = pd.read_csv('BASE_DIR/Audio Data/full_audio_dataset.csv')

def get_audio_path(row):
    filename = row['audio_file']
    voice = row['voice_type'].strip().lower()
    lang = row['language'].strip().lower()

    base_real_ar = [
        'BASE_DIR/Audio Data/Real audio/AlarabyTvSy/AlarabyTvSy',
        'BASE_DIR/Audio Data/Real audio/TelevisionSyria/TelevisionSyria',
        'BASE_DIR/Audio Data/Real audio/mixed_bilingual_files'
    ]
    base_real_en = [
        'BASE_DIR/Audio Data/Real audio/cnn/cnn'
    ]
    base_fake_ar = [
        'BASE_DIR/Audio Data/generated data /VitsTTS/fake',
        'BASE_DIR/Audio Data/generated data /piper_tts/fake'
    ]
    base_fake_en = [
        'BASE_DIR/Audio Data/generated data /tts_outputs_v2/fake'
    ]

    paths_to_try = []
    if voice == 'real' and lang == 'ar':
        paths_to_try = base_real_ar
    elif voice == 'real' and lang == 'en':
        paths_to_try = base_real_en
    elif voice == 'fake' and lang == 'ar':
        paths_to_try = base_fake_ar
    elif voice == 'fake' and lang == 'en':
        paths_to_try = base_fake_en

    for base_path in paths_to_try:
        full_path = os.path.join(base_path, filename)
        if os.path.exists(full_path):
            return full_path
    return None  

df['full_path'] = df.apply(get_audio_path, axis=1)


durations = []
for path in df['full_path']:
    try:
        y, sr = librosa.load(path, sr=16000)
        duration = librosa.get_duration(y=y, sr=sr)
    except:
        duration = None
    durations.append(duration)

df['duration'] = durations


print("🔹 Real Voices Stats:")
print(df[df['voice_type'] == 'real']['duration'].describe())

print("\n🔸 Fake Voices Stats:")
print(df[df['voice_type'] == 'fake']['duration'].describe())

# رسم histogram
plt.figure(figsize=(12, 6))
plt.hist(df[df['voice_type'] == 'real']['duration'], bins=30, alpha=0.6, label='Real Voices')
plt.hist(df[df['voice_type'] == 'fake']['duration'], bins=30, alpha=0.6, label='Fake Voices')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.title('Distribution of Audio Durations (Real vs Fake)')
plt.legend()
plt.grid(True)
plt.show()


# **معالجة الأصوات وتوحيدها إلى 5 ثواني**
# تقطيع الأصوات الطويلة إلى مقاطع مدتها 5 ثواني.
# تمديد الأصوات القصيرة بإضافة padding.

In [ ]:
import os
import pandas as pd
import librosa
import soundfile as sf


df = pd.read_csv('BASE_DIR/Audio Data/prepared_voice_dataset_fixed.csv')


base_output_dir = "/content/processed_voice_dataset2"
os.makedirs(base_output_dir, exist_ok=True)


TARGET_SR = 16000
TARGET_DURATION = 5  
TARGET_LENGTH = TARGET_SR * TARGET_DURATION


def process_and_save_chunks(row):
    chunks = []
    file_path = row['audio_path']
    original_filename = os.path.splitext(row['audio_file'])[0]

    if not os.path.exists(file_path):
        return []

    try:
        y, sr = librosa.load(file_path, sr=TARGET_SR)
    except:
        return []


    path_parts = file_path.split("/")
    if len(path_parts) < 3:
        return []

    model_folder = path_parts[-3]  
    voice_type_folder = path_parts[-2] 
    original_folder = os.path.join(model_folder, voice_type_folder)  


    final_folder = os.path.join(base_output_dir, original_folder, original_filename)
    os.makedirs(final_folder, exist_ok=True)

    total_length = len(y)
    num_chunks = total_length // TARGET_LENGTH

    if total_length < TARGET_LENGTH:
        
        padded = librosa.util.fix_length(data=y, size=TARGET_LENGTH)
        chunk_filename = f"{original_filename}_padded.wav"
        chunk_path = os.path.join(final_folder, chunk_filename)
        sf.write(chunk_path, padded, TARGET_SR)
        chunks.append({
            'chunk_path': chunk_path,
            'label': row['label'],
            'original_audio_file': row['audio_file'],
            'chunk_index': 0
        })

    else:
        
        for i in range(num_chunks):
            start = i * TARGET_LENGTH
            end = start + TARGET_LENGTH
            chunk = y[start:end]
            chunk_filename = f"{original_filename}_chunk{i}.wav"
            chunk_path = os.path.join(final_folder, chunk_filename)
            sf.write(chunk_path, chunk, TARGET_SR)
            chunks.append({
                'chunk_path': chunk_path,
                'label': row['label'],
                'original_audio_file': row['audio_file'],
                'chunk_index': i
            })

    return chunks


all_chunks = []
for _, row in df.iterrows():
    chunks = process_and_save_chunks(row)
    all_chunks.extend(chunks)


processed_df = pd.DataFrame(all_chunks)
processed_df.to_csv("/content/processed_audio_dataset2.csv", index=False)
print(" تم الحفظ بنجاح في مجلدات حسب النموذج ونوع الصوت داخل processed_audio_dataset.csv")


In [ ]:
processed_df

In [ ]:
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import os


df = pd.read_csv('/content/processed_audio_dataset2.csv')


durations = []
for path in df['chunk_path']:
    try:
        y, sr = librosa.load(path, sr=16000)
        duration = librosa.get_duration(y=y, sr=sr)
        durations.append(duration)
    except:
        durations.append(None)


df['duration'] = durations


plt.figure(figsize=(10, 5))
plt.hist(df['duration'].dropna(), bins=50, color='skyblue', edgecolor='black')
plt.title('⏱ Distribution of Audio Chunk Durations')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.grid(True)
plt.show()


print("📊 إحصائيات المدة:")
print(df['duration'].describe())


# عدد المقاطع لكل ملف صوتي أصلي (Original Audio File)
# هذا يوضح إذا كان بعض الملفات الأصلية أنتجت عددًا ضخمًا من المقاطع، مما قد يؤدي إلى انحياز للنموذج

In [ ]:
import seaborn as sns

df = pd.read_csv('/content/processed_audio_dataset2.csv')
plt.figure(figsize=(12, 5))
chunk_counts = df['original_audio_file'].value_counts().head(30)
sns.barplot(x=chunk_counts.index, y=chunk_counts.values)
plt.xticks(rotation=90)
plt.title("Top 30 Original Audio Files by Number of Chunks")
plt.ylabel("Number of Chunks")
plt.xlabel("Original Audio File")
plt.grid(True)
plt.tight_layout()
plt.show()


# نسبة مساهمة كل مصدر (vits, piper, tts_outputs, cnn...) في عدد العينات
#  هل هناك مصدر TTS واحد يهيمن على باقي البيانات؟ هل نموذج CNN الحقيقي مُمثل بما يكفي؟

In [ ]:
df = pd.read_csv('/content/processed_audio_dataset2.csv')

df['source_model'] = df['chunk_path'].apply(lambda x: x.split('/')[3])  # vits / piper / tts_outputs_v2 / cnn / ...
model_counts = df['source_model'].value_counts()

plt.figure(figsize=(6, 4))
model_counts.plot(kind='bar', color='cornflowerblue')
plt.title("Number of Chunks per Voice Source (Model)")
plt.xlabel("Voice Source")
plt.ylabel("Number of Chunks")
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv('/content/processed_audio_dataset2.csv')


df['source_model'] = df['chunk_path'].apply(lambda x: x.split('/')[3]) 


counts = df.groupby(['source_model', 'label']).size().unstack(fill_value=0)


proportions = counts.div(counts.sum(axis=1), axis=0) * 100


proportions.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Paired')
plt.title("🔍 Percentage of Real vs Fake Chunks per Voice Source")
plt.ylabel("Percentage (%)")
plt.xlabel("Voice Source")
plt.legend(["Fake (0)", "Real (1)"])
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


print("📊 نسب real/fake لكل مصدر:")
display(proportions)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv('/content/processed_audio_dataset2.csv')


plt.figure(figsize=(8, 5))
df['label'].value_counts().sort_index().plot(kind='bar', color=['orange', 'green'])
plt.xticks([0, 1], ['Fake Voices (0)', 'Real Voices (1)'])
plt.title("🔊 Number of Audio Chunks per Voice Type")
plt.ylabel("Count")
plt.grid(axis='y')
plt.show()


print("📊 إجمالي المقاطع:")
print(df['label'].value_counts())
print("\n✅ نسبة التوازن:")
print(df['label'].value_counts(normalize=True) * 100)


# **تحويل البيانات إلى Dataset**

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd


df = pd.read_csv('/content/processed_audio_dataset2.csv')


from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)


train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))


dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})


dataset


# **تحويل الملفات الصوتية إلى input_values باستخدام Wav2Vec2Processor.**

# تحميل الملفات الصوتية وتمريريها إلى Wav2Vec2Processor لتحويلها إلى تمثيل رقمي (input_values) مع padding وتوحيد الدقة.

In [ ]:
import os
import gc
import torch
import librosa
import numpy as np
from datasets import load_from_disk, concatenate_datasets

from transformers import Wav2Vec2FeatureExtractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-large-xlsr-53")


In [ ]:

train_dataset = dataset["train"]
batch_size = 1000                      
num_batches = len(train_dataset) // batch_size + 1
output_dir = "/content/processed_batches"
os.makedirs(output_dir, exist_ok=True)

In [ ]:

def preprocess_batch(batch):
    input_values = []
    attention_masks = []

    for path in batch["chunk_path"]:
        try:
            speech_array, _ = librosa.load(path, sr=16000, dtype=np.float32)
            with torch.no_grad():
                inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt", padding="longest")
                input_values.append(inputs.input_values[0])
                attention_masks.append(inputs.attention_mask[0])
            del speech_array, inputs
        except Exception as e:
            print(f"❌ Error processing file: {path} -> {e}")
            input_values.append(torch.zeros(1))
            attention_masks.append(torch.zeros(1))

        gc.collect()

    batch["input_values"] = input_values
    batch["attention_mask"] = attention_masks
    return batch


for i in range(num_batches):
    print(f"\n🔄 Batch {i+1}/{num_batches}")
    start = i * batch_size
    end = min((i + 1) * batch_size, len(train_dataset))
    batch = train_dataset.select(range(start, end))

    processed_batch = batch.map(
        preprocess_batch,
        batched=True,
        batch_size=8,  
        desc=f"🔧 Processing batch {i+1}/{num_batches}"
    )

   
    batch_save_path = os.path.join(output_dir, f"train_batch_{i}")
    processed_batch.save_to_disk(batch_save_path)

    
    del batch, processed_batch
    gc.collect()

In [ ]:

all_batches = []
for i in range(num_batches):
    path = os.path.join(output_dir, f"train_batch_{i}")
    ds = load_from_disk(path)
    all_batches.append(ds)


final_train_dataset = concatenate_datasets(all_batches)


dataset["train"] = final_train_dataset


print(f" عدد العينات بعد الدمج: {len(dataset['train'])}")


In [ ]:
final_train_dataset

# **هلا لل validation  بنفس الطريقه**

In [ ]:

val_dataset = dataset["validation"]
val_batch_size = 500
val_num_batches = len(val_dataset) // val_batch_size + 1

val_output_dir = "/content/processed_val_batches"
os.makedirs(val_output_dir, exist_ok=True)


for i in range(val_num_batches):
    print(f"\ [VAL] Batch {i+1}/{val_num_batches}")
    start = i * val_batch_size
    end = min((i + 1) * val_batch_size, len(val_dataset))
    batch = val_dataset.select(range(start, end))

    processed_batch = batch.map(
        preprocess_batch,
        batched=True,
        batch_size=8,
        desc=f" Processing validation batch {i+1}/{val_num_batches}"
    )

 
    batch_path = os.path.join(val_output_dir, f"val_batch_{i}")
    processed_batch.save_to_disk(batch_path)

    del batch, processed_batch
    gc.collect()


In [ ]:

val_batches = []
for i in range(val_num_batches):
    path = os.path.join(val_output_dir, f"val_batch_{i}")
    ds = load_from_disk(path)
    val_batches.append(ds)


final_val_dataset = concatenate_datasets(val_batches)
dataset["validation"] = final_val_dataset

print(f" عدد عينات التحقق بعد الدمج: {len(dataset['validation'])}")


# save

In [ ]:

drive_path = "BASE_DIR/audio_final_datasets"
os.makedirs(drive_path, exist_ok=True)


train_save_path = os.path.join(drive_path, "final_train_dataset")
final_train_dataset.save_to_disk(train_save_path)
print(" تم حفظ final_train_dataset بنجاح.")


val_save_path = os.path.join(drive_path, "final_val_dataset")
final_val_dataset.save_to_disk(val_save_path)
print(" تم حفظ final_val_dataset بنجاح.")


# load

In [ ]:
from datasets import load_from_disk


train_dataset = load_from_disk("BASE_DIR/audio_final_datasets/final_train_dataset")
val_dataset = load_from_disk("BASE_DIR/audio_final_datasets/final_val_dataset")


from datasets import DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})


# **توزيع عدد العينات لكل فئة (real vs fake)**

In [ ]:
import matplotlib.pyplot as plt

label_counts = dataset["train"].features["label"].names if hasattr(dataset["train"].features["label"], "names") else None
dataset["train"].to_pandas()["label"].value_counts().plot(kind='bar', color=["green", "orange"])
plt.title("Distribution of Labels in Training Data (Real vs Fake)")
plt.xlabel("Label (0=Fake, 1=Real)")
plt.ylabel("Count")
plt.grid(True)
plt.show()


# **توزيع أطوال input_values**

# للتأكد أن كل المقاطع تم توحيد طولها (مثلاً 5 ثوانٍ = 80000 عينة عند sr=16000)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random


subset_size = 300
indices = random.sample(range(len(dataset["train"])), subset_size)
subset = [dataset["train"][i]["input_values"] for i in indices]


lengths = [len(x) for x in subset]


plt.hist(lengths, bins=20, color='skyblue')
plt.title("Distribution of Input Value Lengths (Sampled)")
plt.xlabel("Length (#samples)")
plt.ylabel("Count")
plt.grid(True)
plt.show()


print(f"✅ Sampled {subset_size} entries:")
print("Min:", np.min(lengths))
print("Max:", np.max(lengths))
print("Mean:", np.mean(lengths))


# **رسم عينة صوتية واحدة (input_values)**

In [ ]:
import matplotlib.pyplot as plt
example = dataset["train"][0]
plt.plot(example["input_values"])
plt.title("Waveform of Sample Audio")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()


# **توزيع أطوال attention_mask**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

subset_size = 300
indices = random.sample(range(len(dataset["train"])), subset_size)


lengths = [int(sum(dataset["train"][i]["attention_mask"])) for i in indices]


plt.hist(lengths, bins=20, color='lightcoral')
plt.title("Distribution of Attention Mask Lengths (Sampled)")
plt.xlabel("Length (non-padded tokens)")
plt.ylabel("Count")
plt.grid(True)
plt.show()


print(f"✅ Sampled {subset_size} entries:")
print("Min:", np.min(lengths))
print("Max:", np.max(lengths))
print("Mean:", np.mean(lengths))


# **مقارنة موجتين من فئتين مختلفتين (real vs fake)**

In [ ]:
import random
import matplotlib.pyplot as plt


def get_sample_by_label(label_value):
    while True:
        index = random.randint(0, len(dataset["train"]) - 1)
        if dataset["train"][index]["label"] == label_value:
            return dataset["train"][index]


real_sample = get_sample_by_label(1)
fake_sample = get_sample_by_label(0)


plt.figure(figsize=(12, 3))

plt.subplot(1, 2, 1)
plt.plot(real_sample["input_values"])
plt.title("✅ Real Voice Sample")

plt.subplot(1, 2, 2)
plt.plot(fake_sample["input_values"])
plt.title("❌ Fake Voice Sample")

plt.tight_layout()
plt.show()


# رسم الطيف الترددي (Spectrogram)

# يُظهر الفرق في الطاقة والترددات عبر الزمن
# الأصوات الحقيقية غالبًا تكون أكثر تعقيدًا

In [ ]:
import librosa.display
import numpy as np


def plot_spectrogram(waveform, title):
    plt.figure(figsize=(10, 3))
    D = librosa.amplitude_to_db(np.abs(librosa.stft(np.array(waveform))), ref=np.max)
    librosa.display.specshow(D, sr=16000, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_spectrogram(real_sample["input_values"], "✅ Real Voice - Spectrogram")
plot_spectrogram(fake_sample["input_values"], "❌ Fake Voice - Spectrogram")


# رسم توزيع القيم الصوتية (Amplitude Histogram)

# يوضح الاختلاف في ديناميكية الصوت ومستوى الضغط الصوتي

In [ ]:
plt.figure(figsize=(10, 3))

plt.hist(real_sample["input_values"], bins=100, alpha=0.6, label="Real", color="green")
plt.hist(fake_sample["input_values"], bins=100, alpha=0.6, label="Fake", color="red")

plt.title("Distribution of Amplitudes (Real vs Fake)")
plt.xlabel("Amplitude Value")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# مقارنة FFT (Fast Fourier Transform)
# لعرض توزيع الترددات الصوتية (غالبًا أكثر تعقيدًا في الصوت الحقيقي)

In [ ]:
from scipy.fft import fft


real_fft = np.abs(fft(real_sample["input_values"]))[:4000]
fake_fft = np.abs(fft(fake_sample["input_values"]))[:4000]

plt.figure(figsize=(12, 3))
plt.plot(real_fft, label="Real", color="blue")
plt.plot(fake_fft, label="Fake", color="orange")
plt.title("FFT Comparison (Real vs Fake)")
plt.xlabel("Frequency Bin")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
!pip install transformers datasets evaluate accelerate --quiet

In [ ]:
!pip install -U transformers


In [ ]:
import pandas as pd


df = pd.read_csv("/content/processed_audio_dataset2.csv")


def detect_language(path):
    if "cnn/cnn" in path.lower():
        return "en"
    elif "alarabytvsy" in path.lower() or "televisionsyria" in path.lower():
        return "ar"
    elif "mixed_bilingual_files" in path.lower():
        return "mixed"
    elif "tts_outputs_v2" in path.lower():
        return "en"
    elif "piper_tts" in path.lower() or "vits" in path.lower():
        return "ar"
    else:
        return "unknown"


df["language"] = df["chunk_path"].apply(detect_language)


df.to_csv("/content/processed_audio_dataset2.csv", index=False)
print(" تم تحديد اللغة وحفظ التحديث في نفس الملف.")


In [ ]:
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd


df = pd.read_csv("BASE_DIR/Audio Data/processed_audio_dataset2.csv")


df['language'] = df['language'].replace({'mixed': 'ar'})



sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)


plt.figure()
sns.countplot(data=df, x="language", palette="Set2")
plt.title("Number of Samples per Language")
plt.xlabel("Language")
plt.ylabel("Sample Count")
plt.show()


plt.figure()
sns.countplot(data=df, x="label", palette="Set1")
plt.title("Overall Distribution of Real vs Fake")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Sample Count")
plt.show()


plt.figure()
sns.countplot(data=df, x="language", hue="label", palette="husl")
plt.title("Real vs Fake Samples per Language")
plt.xlabel("Language")
plt.ylabel("Sample Count")
plt.legend(title="Label", labels=["Fake", "Real"])
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(data=df, x="language", palette="Set2")
plt.title("Number of Samples per Language (after merging 'mixed' into Arabic)")
plt.xlabel("Language")
plt.ylabel("Sample Count")
plt.show()


In [ ]:

import pandas as pd
df = pd.read_csv("BASE_DIR/Audio Data/processed_audio_dataset2.csv")


df['language'] = df['language'].replace("mixed", "ar")


df_ar = df[df["language"] == "ar"]
df_en = df[df["language"] == "en"]


df_ar.to_csv("/content/arabic_audio_dataset.csv", index=False)
df_en.to_csv("/content/english_audio_dataset.csv", index=False)

print(" Saved Arabic and English datasets separately.")


In [ ]:
df_ar

In [ ]:
df_en

# **starting with Arabic Language**

In [ ]:
!pip install datasets


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd


df_full = pd.read_csv("BASE_DIR/Audio Data/processed_audio_dataset2.csv")


df_ar = df_full[df_full['language'] == 'ar'].reset_index(drop=True)


train_df_ar, val_df_ar = train_test_split(df_ar, test_size=0.1, random_state=42, stratify=df_ar['label'])

# Convert to HuggingFace Datasets
train_dataset_ar = Dataset.from_pandas(train_df_ar)
val_dataset_ar = Dataset.from_pandas(val_df_ar)

# Group in DatasetDict
dataset_ar = DatasetDict({
    "train": train_dataset_ar,
    "validation": val_dataset_ar
})


dataset_ar.save_to_disk("/content/processed_voice_dataset2/arabic_audio_dataset")
print("✅ Arabic dataset saved to disk successfully!")


In [ ]:
from datasets import Dataset, DatasetDict


train_dataset_ar = Dataset.from_pandas(train_df_ar)
val_dataset_ar = Dataset.from_pandas(val_df_ar)


dataset_ar = DatasetDict({
    "train": train_dataset_ar,
    "validation": val_dataset_ar
})

print(dataset_ar)


In [ ]:
!pip install torch librosa datasets transformers

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd
from sklearn.model_selection import train_test_split
import os, gc, torch, librosa
import numpy as np
from transformers import Wav2Vec2FeatureExtractor
from datasets import load_from_disk, concatenate_datasets


csv_path = "/content/arabic_audio_dataset.csv"  


df = pd.read_csv(csv_path)
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)


train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
dataset = DatasetDict({'train': train_dataset, 'validation': val_dataset})


feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-large-xlsr-53")


train_dataset = dataset["train"]
batch_size = 1000
num_batches = len(train_dataset) // batch_size + 1
output_dir = "/content/processed_batches"
os.makedirs(output_dir, exist_ok=True)


def preprocess_batch(batch):
    input_values = []
    attention_masks = []

    for path in batch["chunk_path"]:
        try:
            speech_array, _ = librosa.load(path, sr=16000, dtype=np.float32)
            with torch.no_grad():
                inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt", padding="longest")
                input_values.append(inputs.input_values[0])
                attention_masks.append(inputs.attention_mask[0])
        except Exception as e:
            print(f"❌ Error processing file: {path} -> {e}")
            input_values.append(torch.zeros(1))
            attention_masks.append(torch.zeros(1))
        gc.collect()

    batch["input_values"] = input_values
    batch["attention_mask"] = attention_masks
    return batch

In [ ]:

for i in range(num_batches):
    print(f"\ Batch {i+1}/{num_batches}")
    start = i * batch_size
    end = min((i + 1) * batch_size, len(train_dataset))
    batch = train_dataset.select(range(start, end))

    processed_batch = batch.map(
        preprocess_batch,
        batched=True,
        batch_size=8,
        desc=f" Processing batch {i+1}/{num_batches}"
    )

    batch_save_path = os.path.join(output_dir, f"train_batch_{i}")
    processed_batch.save_to_disk(batch_save_path)

    del batch, processed_batch
    gc.collect()

In [ ]:

all_batches = []
for i in range(num_batches):
    path = os.path.join(output_dir, f"train_batch_{i}")
    ds = load_from_disk(path)
    all_batches.append(ds)

final_train_dataset = concatenate_datasets(all_batches)
dataset["train"] = final_train_dataset
print(f" عدد عينات التدريب: {len(dataset['train'])}")


In [ ]:

val_dataset = dataset["validation"]
val_batch_size = 500
val_num_batches = len(val_dataset) // val_batch_size + 1
val_output_dir = "/content/processed_val_batches"
os.makedirs(val_output_dir, exist_ok=True)

for i in range(val_num_batches):
    print(f"\n [VAL] Batch {i+1}/{val_num_batches}")
    start = i * val_batch_size
    end = min((i + 1) * val_batch_size, len(val_dataset))
    batch = val_dataset.select(range(start, end))

    processed_batch = batch.map(
        preprocess_batch,
        batched=True,
        batch_size=8,
        desc=f" Processing validation batch {i+1}/{val_num_batches}"
    )

    batch_path = os.path.join(val_output_dir, f"val_batch_{i}")
    processed_batch.save_to_disk(batch_path)

    del batch, processed_batch
    gc.collect()

In [ ]:

val_batches = []
for i in range(val_num_batches):
    path = os.path.join(val_output_dir, f"val_batch_{i}")
    ds = load_from_disk(path)
    val_batches.append(ds)

final_val_dataset = concatenate_datasets(val_batches)
dataset["validation"] = final_val_dataset
print(f" عدد عينات التحقق بعد الدمج: {len(dataset['validation'])}")


In [ ]:

drive_path = "BASE_DIR/audio_final_datasets"
os.makedirs(drive_path, exist_ok=True)

train_save_path = os.path.join(drive_path, "final_train_dataset")
final_train_dataset.save_to_disk(train_save_path)
print(" تم حفظ final_train_dataset بنجاح.")

val_save_path = os.path.join(drive_path, "final_val_dataset")
final_val_dataset.save_to_disk(val_save_path)
print(" تم حفظ final_val_dataset بنجاح.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


df_train = dataset["train"].to_pandas()

plt.figure(figsize=(6, 4))
sns.countplot(data=df_train, x="label", palette="Set2")
plt.title("Distribution of Real vs Fake in Training Data")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Sample Count")
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df_train, x="language", hue="label", palette="Set1")
plt.title("Real vs Fake Samples per Language (Train Data)")
plt.xlabel("Language")
plt.ylabel("Sample Count")
plt.legend(title="Label", labels=["Fake", "Real"])
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.show()


In [ ]:
from datasets import load_from_disk


drive_path = "BASE_DIR/audio_final_datasets"
train_save_path = f"{drive_path}/final_train_dataset"
val_save_path = f"{drive_path}/final_val_dataset"


final_train_dataset = load_from_disk(train_save_path)
final_val_dataset = load_from_disk(val_save_path)

from datasets import DatasetDict
dataset = DatasetDict({
    'train': final_train_dataset,
    'validation': final_val_dataset
})


print(f" Loaded train dataset: {len(dataset['train'])} samples")
print(f" Loaded validation dataset: {len(dataset['validation'])} samples")


In [ ]:
!pip install transformers evaluate

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("elgeish/wav2vec2-large-xlsr-53-arabic")


In [ ]:
import torch


def collate_fn(batch):
    input_values = [item["input_values"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["label"] for item in batch]


    padded = processor.pad(
        {
            "input_values": input_values,
            "attention_mask": attention_mask
        },
        return_tensors="pt"
    )

    return {
        "input_values": padded["input_values"],
        "attention_mask": padded["attention_mask"],
        "labels": torch.tensor(labels, dtype=torch.long)
    }


In [ ]:
from transformers import AutoModelForAudioClassification


model = AutoModelForAudioClassification.from_pretrained(
    "elgeish/wav2vec2-large-xlsr-53-arabic",
    num_labels=2  
)


In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

def compute_metrics(pred):
    logits = pred.predictions
    preds = np.argmax(logits, axis=-1)
    labels = pred.label_ids

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = np.mean(preds == labels)
    auc = roc_auc_score(labels, logits[:, 1])

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }




In [ ]:

training_args = TrainingArguments(
    output_dir="./wav2vec2-arabic-fake-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1
)


trainer = Trainer(
    model=model,  
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

In [ ]:

trainer.train()


In [ ]:

save_path = "BASE_DIR/audio_final_datasets/final_arabic_model"


import os
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(" Model and processor saved successfully.")


# **Training Loss vs. Validation Loss**

In [ ]:
train_loss = []
eval_loss = []
steps = []

for log in trainer.state.log_history:
    if 'loss' in log:
        train_loss.append(log['loss'])
        steps.append(log['step'])
    if 'eval_loss' in log:
        eval_loss.append(log['eval_loss'])


plt.figure(figsize=(7, 5))
plt.plot(steps[:len(train_loss)], train_loss, label="Training Loss", color='blue')
if eval_loss:
    plt.plot(steps[:len(eval_loss)], eval_loss, label="Validation Loss", color='orange')

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.grid(True)
plt.legend()
plt.show()


# ***Confusion Matrix***

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np


preds = trainer.predict(dataset['validation'])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()


# ***ROC Curve & AUC Score***

In [ ]:
from sklearn.metrics import roc_curve, auc


y_probs = preds.predictions[:, 1]

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)


plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


# ***Classification Report***

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


# ***Testing on validation set***

# تنفيذ التنبؤات على validation data

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


val_inputs = dataset['validation'].remove_columns(['label'])
val_labels = dataset['validation']['label']


predictions = trainer.predict(dataset['validation'])
preds = np.argmax(predictions.predictions, axis=1)


print("📋 Classification Report:\n")
print(classification_report(val_labels, preds, digits=4))


cm = confusion_matrix(val_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix (Validation Set)")
plt.grid(False)
plt.show()


# **رسم منحنى ROC Curve**

In [ ]:

prob_real = predictions.predictions[:, 1]

fpr, tpr, thresholds = roc_curve(val_labels, prob_real)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Validation Set)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


# **2**

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback


training_args = TrainingArguments(
    output_dir="./wav2vec2-arabic-fake-news-v2",
    eval_strategy="steps",            # تقييم كل عدد معين من الخطوات
    save_strategy="steps",                  # حفظ النموذج بناءً على خطوات
    logging_strategy="steps",               # تسجيل الأحداث أثناء الخطوات
    eval_steps=500,                         # تقييم كل 500 خطوة
    save_steps=500,                         # حفظ النموذج كل 500 خطوة
    logging_steps=500,                      # تسجيل الأحداث كل 500 خطوة
    learning_rate=3e-5,
    per_device_train_batch_size=8,          # تقليل حجم الدفعة يقلل من overfitting
    per_device_eval_batch_size=8,
    num_train_epochs=3,                     # عدد مناسب مع early stopping
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True                               # لو تعمل على GPU لدعم السرعة
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:

trainer.train()


In [ ]:
import matplotlib.pyplot as plt


train_logs = trainer.state.log_history


train_steps, train_losses, val_steps, val_losses = [], [], [], []

for log in train_logs:
    if "loss" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log:
        val_steps.append(log["step"])
        val_losses.append(log["eval_loss"])


plt.figure(figsize=(8, 5))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(val_steps, val_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


predictions = trainer.predict(dataset["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

# احتمالات Real
y_probs = predictions.predictions[:, 1]  # class 1 = Real
fpr, tpr, thresholds = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.4f})", color="darkorange", linewidth=2)
plt.plot([0, 1], [0, 1], color="navy", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Validation Set)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print("📋 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


In [ ]:
import os


save_path = "BASE_DIR/audio_final_datasets/final_arabic_model_v2"
os.makedirs(save_path, exist_ok=True)


model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("✅ Model and processor saved successfully to:", save_path)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


labels = dataset['train']['label']

plt.figure(figsize=(6, 4))
sns.countplot(x=labels)
plt.title("Distribution of Labels in Training Set")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Sample Count")
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd

df_train = dataset['train'].to_pandas()

plt.figure(figsize=(8, 5))
sns.countplot(data=df_train, x="language", hue="label", palette="Set1")
plt.title("Label Distribution per Language (Train Set)")
plt.xlabel("Language")
plt.ylabel("Sample Count")
plt.legend(title="Label", labels=["Fake", "Real"])
plt.grid(True)
plt.show()


3

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

labels = dataset["train"]["label"]
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32)


In [ ]:
from transformers import AutoProcessor, AutoModelForAudioClassification

model_name = "elgeish/wav2vec2-large-xlsr-53-arabic"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
def collate_fn(batch):
    input_values = [item["input_values"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["label"] for item in batch]

    batch = processor.pad(
        {"input_values": input_values, "attention_mask": attention_mask},
        return_tensors="pt"
    )
    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    auc = roc_auc_score(labels, eval_pred.predictions[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
from transformers import Trainer
import torch.nn.functional as F

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device) if class_weights is not None else None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):  
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = F.cross_entropy(logits, labels, weight=self.class_weights)
        return (loss, outputs) if return_outputs else loss


In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./wav2vec2-arabic-balanced-v2",
    eval_strategy="steps",              
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=500,                           
    save_steps=500,
    logging_steps=100,

    learning_rate=2e-5,                         
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,                        
    weight_decay=0.01,
    fp16=True,                                  

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1                          
)


early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3                   
)


In [ ]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    class_weights=class_weights,            
    callbacks=[early_stopping]               
)


In [ ]:
trainer.train()


In [ ]:

save_path = "BASE_DIR/audio_final_datasets/final_balanced_arabic_modelv3"


import os
os.makedirs(save_path, exist_ok=True)


model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("✅ Model and processor saved successfully.")


In [ ]:
from transformers import AutoModelForAudioClassification, AutoProcessor


model_path = "BASE_DIR/Audio Data/audio_final_datasets/final_balanced_arabic_modelv3"


model = AutoModelForAudioClassification.from_pretrained(model_path)
processor = AutoProcessor.from_pretrained(model_path)

print(" Model and processor loaded successfully.")


In [ ]:
from datasets import load_from_disk

val_dataset = load_from_disk("BASE_DIR/Audio Data/audio_final_datasets/final_val_dataset")
print(f" Validation samples: {len(val_dataset)}")


In [ ]:
import torch

def collate_fn(batch):
    input_values = [item["input_values"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["label"] for item in batch]

    padded = processor.pad(
        {"input_values": input_values, "attention_mask": attention_mask},
        return_tensors="pt"
    )

    return {
        "input_values": padded["input_values"],
        "attention_mask": padded["attention_mask"],
        "labels": torch.tensor(labels, dtype=torch.long)
    }


In [ ]:
from transformers import Trainer, TrainingArguments

eval_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=16
)

trainer = Trainer(
    model=model,
    args=eval_args,
    tokenizer=processor,
    data_collator=collate_fn
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


training_data = {
    "Step": [500, 1000, 1500],
    "Training Loss": [0.007700, 0.000000, 0.000000],
    "Validation Loss": [0.002869, 0.000015, 0.000010],
    "Accuracy": [0.998745, 1.000000, 1.000000],
    "Precision": [0.997021, 1.000000, 1.000000],
    "Recall": [1.000000, 1.000000, 1.000000],
    "F1": [0.998508, 1.000000, 1.000000],
    "AUC": [1.000000, 1.000000, 1.000000]
}

df = pd.DataFrame(training_data)
display(df)


plt.figure(figsize=(8,5))
plt.plot(df["Step"], df["Training Loss"], label="Training Loss")
plt.plot(df["Step"], df["Validation Loss"], label="Validation Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8,5))
plt.plot(df["Step"], df["Accuracy"], label="Accuracy")
plt.plot(df["Step"], df["F1"], label="F1 Score")
plt.xlabel("Steps")
plt.ylabel("Score")
plt.title("Accuracy and F1 Score over Steps")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

preds = trainer.predict(val_dataset)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc


y_probs = preds.predictions[:, 1]
fpr, tpr, thresholds = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.4f})", color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='navy')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Validation Set)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print("📊 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


# **English Section**

In [ ]:
import pandas as pd


df_all = pd.read_csv("/content/processed_audio_dataset2.csv")


df_en = df_all[df_all['language'] == 'en'].reset_index(drop=True)


df_en.to_csv("/content/english_audio_dataset.csv", index=False)

print(f"✅ عدد العينات الإنجليزية: {len(df_en)}")


In [ ]:
df_en

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.countplot(x="label", data=df_en)
plt.title("Distribution of Labels in English Dataset")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Sample Count")
plt.grid(True)
plt.show()


In [ ]:
df_en["model_type"] = df_en["chunk_path"].apply(lambda x: x.split("/")[3])  # استخراج اسم مجلد النموذج

plt.figure(figsize=(6, 4))
sns.countplot(x="model_type", hue="label", data=df_en)
plt.title("Label Distribution per Model (English Audio)")
plt.xlabel("Model Type")
plt.ylabel("Sample Count")
plt.grid(True)
plt.legend(title="Label", labels=["Fake", "Real"])
plt.show()


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd


df_full = pd.read_csv("/content/processed_audio_dataset2.csv")


df_en = df_full[df_full['language'] == 'en'].reset_index(drop=True)


train_df_en, val_df_en = train_test_split(df_en, test_size=0.1, random_state=42, stratify=df_en['label'])


train_dataset_en = Dataset.from_pandas(train_df_en)
val_dataset_en = Dataset.from_pandas(val_df_en)


dataset_en = DatasetDict({
    "train": train_dataset_en,
    "validation": val_dataset_en
})


dataset_en.save_to_disk("/content/processed_voice_dataset2/english_audio_dataset")
print("✅ english dataset saved to disk successfully!")


In [ ]:
dataset_en

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd
from sklearn.model_selection import train_test_split
import os, gc, torch, librosa
import numpy as np
from transformers import Wav2Vec2FeatureExtractor
from datasets import load_from_disk, concatenate_datasets


csv_path = "/content/english_audio_dataset.csv"


df = pd.read_csv(csv_path)
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)


train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
dataset = DatasetDict({'train': train_dataset, 'validation': val_dataset})

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-large-xlsr-53")


train_dataset = dataset["train"]
batch_size = 1000
num_batches = len(train_dataset) // batch_size + 1
output_dir = "/content/processed_batches"
os.makedirs(output_dir, exist_ok=True)


def preprocess_batch(batch):
    input_values = []
    attention_masks = []

    for path in batch["chunk_path"]:
        try:
            speech_array, _ = librosa.load(path, sr=16000, dtype=np.float32)
            with torch.no_grad():
                inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt", padding="longest")
                input_values.append(inputs.input_values[0])
                attention_masks.append(inputs.attention_mask[0])
        except Exception as e:
            print(f"❌ Error processing file: {path} -> {e}")
            input_values.append(torch.zeros(1))
            attention_masks.append(torch.zeros(1))
        gc.collect()

    batch["input_values"] = input_values
    batch["attention_mask"] = attention_masks
    return batch

In [ ]:

for i in range(num_batches):
    print(f"\n🔄 Batch {i+1}/{num_batches}")
    start = i * batch_size
    end = min((i + 1) * batch_size, len(train_dataset))
    batch = train_dataset.select(range(start, end))

    processed_batch = batch.map(
        preprocess_batch,
        batched=True,
        batch_size=8,
        desc=f"🔧 Processing batch {i+1}/{num_batches}"
    )

    batch_save_path = os.path.join(output_dir, f"train_batch_{i}")
    processed_batch.save_to_disk(batch_save_path)

    del batch, processed_batch
    gc.collect()

In [ ]:

all_batches = []
for i in range(num_batches):
    path = os.path.join(output_dir, f"train_batch_{i}")
    ds = load_from_disk(path)
    all_batches.append(ds)

final_train_dataset = concatenate_datasets(all_batches)
dataset["train"] = final_train_dataset
print(f" عدد عينات التدريب: {len(dataset['train'])}")


In [ ]:

val_dataset = dataset["validation"]
val_batch_size = 500
val_num_batches = len(val_dataset) // val_batch_size + 1
val_output_dir = "/content/processed_val_batches"
os.makedirs(val_output_dir, exist_ok=True)

for i in range(val_num_batches):
    print(f"\n [VAL] Batch {i+1}/{val_num_batches}")
    start = i * val_batch_size
    end = min((i + 1) * val_batch_size, len(val_dataset))
    batch = val_dataset.select(range(start, end))

    processed_batch = batch.map(
        preprocess_batch,
        batched=True,
        batch_size=8,
        desc=f" Processing validation batch {i+1}/{val_num_batches}"
    )

    batch_path = os.path.join(val_output_dir, f"val_batch_{i}")
    processed_batch.save_to_disk(batch_path)

    del batch, processed_batch
    gc.collect()

In [ ]:

val_batches = []
for i in range(val_num_batches):
    path = os.path.join(val_output_dir, f"val_batch_{i}")
    ds = load_from_disk(path)
    val_batches.append(ds)

final_val_dataset = concatenate_datasets(val_batches)
dataset["validation"] = final_val_dataset
print(f" عدد عينات التحقق بعد الدمج: {len(dataset['validation'])}")


In [ ]:

drive_path = "BASE_DIR/audioEnglsih_final_datasets"
os.makedirs(drive_path, exist_ok=True)

train_save_path = os.path.join(drive_path, "final_train_dataset")
final_train_dataset.save_to_disk(train_save_path)
print(" تم حفظ final_train_dataset بنجاح.")

val_save_path = os.path.join(drive_path, "final_val_dataset")
final_val_dataset.save_to_disk(val_save_path)
print(" تم حفظ final_val_dataset بنجاح.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv("/content/english_audio_dataset.csv") 


df['source'] = df['chunk_path'].apply(lambda x: x.split("/")[-3]) 


plt.figure(figsize=(10, 6))
sns.countplot(data=df, x="label", hue="source")
plt.title("Distribution of sources per label (real vs fake)")
plt.xlabel("Label (1 = Real, 0 = Fake)")
plt.ylabel("Number of Samples")
plt.legend(title="Source")
plt.grid(True)
plt.show()


In [ ]:
from datasets import load_from_disk


drive_path = "BASE_DIR/audioEnglsih_final_datasets"
train_save_path = f"{drive_path}/final_train_dataset"
val_save_path = f"{drive_path}/final_val_dataset"


final_train_dataset = load_from_disk(train_save_path)
final_val_dataset = load_from_disk(val_save_path)


from datasets import DatasetDict
dataset = DatasetDict({
    'train': final_train_dataset,
    'validation': final_val_dataset
})


print(f"✅ Loaded train dataset: {len(dataset['train'])} samples")
print(f"✅ Loaded validation dataset: {len(dataset['validation'])} samples")


In [ ]:
!pip install transformers evaluate

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

labels = dataset["train"]["label"]
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32)


In [ ]:
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

model_name = "facebook/wav2vec2-large-xlsr-53"
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name, num_labels=2)
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)


In [ ]:
def collate_fn(batch):
    input_values = [item["input_values"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["label"] for item in batch]

    padded = processor.pad(
        {"input_values": input_values, "attention_mask": attention_mask},
        return_tensors="pt"
    )

    return {
        "input_values": padded["input_values"],
        "attention_mask": padded["attention_mask"],
        "labels": torch.tensor(labels, dtype=torch.long)
    }


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="macro")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-english-fake-news",
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=100,
    save_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    save_total_limit=1
)


In [ ]:
from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)


In [ ]:
from transformers import Trainer

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=self.class_weights)
        return (loss, outputs) if return_outputs else loss



In [ ]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[early_stopping]
)

trainer.train()


In [ ]:
import matplotlib.pyplot as plt


steps = [100, 200, 300, 400, 500, 600, 700]
val_loss = [0.5659, 0.3729, 0.3919, 0.4208, 0.2185, 0.1800, 0.2829]
train_loss = [None, None, None, None, 0.3170, 0.3170, 0.3170]

plt.figure(figsize=(8, 4))
plt.plot(steps, val_loss, label="Validation Loss", marker='o')
plt.plot(steps[-3:], train_loss[-3:], label="Training Loss", marker='x')  
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
acc =    [0.8259, 0.8928, 0.8973, 0.9196, 0.9426, 0.9687, 0.9592]
prec =   [0.8454, 0.9013, 0.9127, 0.9269, 0.9680, 0.9727, 0.9651]
recall = [0.8169, 0.8787, 0.8970, 0.9154, 0.9518, 0.9663, 0.9567]
f1 =     [0.8198, 0.8900, 0.8948, 0.9184, 0.9603, 0.9684, 0.9593]

plt.figure(figsize=(10, 5))
plt.plot(steps, acc, label="Accuracy", marker='o')
plt.plot(steps, prec, label="Precision", marker='o')
plt.plot(steps, recall, label="Recall", marker='o')
plt.plot(steps, f1, label="F1 Score", marker='o')
plt.xlabel("Step")
plt.ylabel("Score")
plt.title("Evaluation Metrics over Steps")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from evaluate import load

accuracy = load("accuracy")
precision = load("precision")
recall = load("recall")
f1 = load("f1")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np


preds = trainer.predict(dataset["validation"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix (Validation Set)")
plt.grid(False)
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


# **2**

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./wav2vec2-english-fake-news-v2",
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)


In [ ]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[early_stopping]
)


In [ ]:
trainer.train()


In [ ]:
import matplotlib.pyplot as plt


steps = [100, 200, 300, 400, 500, 600, 700,800]
val_loss = [0.463952, 0.225049, 0.069476, 0.307667, 0.058479, 0.198386, 0.241743,0.219285]
train_loss = [0.091600, 0.053000, 0.028400, 0.019800, 0.013700, 0.013400, 0.030900,0.042100]


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(steps, val_loss, label="Validation Loss", marker='o')
plt.plot(steps[-3:], train_loss[-3:], label="Training Loss", marker='x')  # رسم آخر 3 نقاط فقط إن توفرت
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt


history = trainer.state.log_history


steps = [x['step'] for x in history if 'eval_loss' in x]
accuracy = [x['eval_accuracy'] for x in history if 'eval_accuracy' in x]
precision = [x['eval_precision'] for x in history if 'eval_precision' in x]
recall = [x['eval_recall'] for x in history if 'eval_recall' in x]
f1 = [x['eval_f1'] for x in history if 'eval_f1' in x]


plt.figure(figsize=(10, 6))
plt.plot(steps, accuracy, label="Accuracy")
plt.plot(steps, precision, label="Precision")
plt.plot(steps, recall, label="Recall")
plt.plot(steps, f1, label="F1 Score")
plt.xlabel("Step")
plt.ylabel("Score")
plt.title(" Accuracy / Precision / Recall / F1 per Step")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np


preds = trainer.predict(dataset["validation"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)
y_probs = preds.predictions[:, 1] 


accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)


print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1 Score: {f1:.4f}")
print(f"✅ ROC-AUC: {auc:.4f}")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


cm = confusion_matrix(y_true, y_pred)


disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()


In [ ]:

save_path = "BASE_DIR/audioEnglish_final_datasets/final_balanced_english_model"

import os
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print("✅ Model and processor saved successfully.")


# **3**

In [ ]:
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    num_labels=2,
    gradient_checkpointing=True,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    classifier_proj_size=256,
)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-english-fake-news",
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    warmup_ratio=0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True
)


In [ ]:
from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

labels = dataset['train']['label']
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32)


In [ ]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[early_stopping]
)

trainer.train()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


data = {
    "Step": [200, 400, 600, 800, 1000, 1200, 1400,1600,1800],
    "Training Loss": [0.689800, 0.473300, 0.196500, 0.095100, 0.050900, 0.027000, 0.013100,0.009000,0.011800],
    "Validation Loss": [0.666215, 0.297412, 0.111922, 0.119759, 0.012026, 0.001587, 0.009297,0.046630,0.115674],
    "Accuracy": [0.812500,0.937500,0.968750,0.973214,0.991071,1.000000,0.995536,0.991071,0.982143],
    "Precision": [0.839459,0.939909,0.972441,0.976190,0.991803,1.000000,0.995868,0.991803,0.983871],
    "Recall": [0.801923,0.935256,0.966346,0.971154,0.990385,1.000000,0.995192,0.990385,0.980769],
    "F1": [0.804245,0.936891,0.968417,0.972953,0.991013,1.000000,0.995510,0.991013,0.981999],
}

df = pd.DataFrame(data)


plt.figure(figsize=(10, 4))
plt.plot(df["Step"], df["Training Loss"], label="Training Loss", marker='o')
plt.plot(df["Step"], df["Validation Loss"], label="Validation Loss", marker='o')
plt.title("Training vs Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()


plt.figure(figsize=(10, 4))
plt.plot(df["Step"], df["Accuracy"], label="Accuracy", marker='o')
plt.plot(df["Step"], df["Precision"], label="Precision", marker='o')
plt.plot(df["Step"], df["Recall"], label="Recall", marker='o')
plt.plot(df["Step"], df["F1"], label="F1 Score", marker='o')
plt.title("Validation Metrics Over Training Steps")
plt.xlabel("Step")
plt.ylabel("Score")
plt.ylim(0.8, 1.0)
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np


preds = trainer.predict(dataset["validation"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)


cm = confusion_matrix(y_true, y_pred, labels=[0, 1])  # [0 = Fake, 1 = Real]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()


tn, fp, fn, tp = cm.ravel()


print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")


In [ ]:

save_path = "BASE_DIR/final_balanced_english_model_complete"

trainer.save_model(save_path)


processor.save_pretrained(save_path)
